In [22]:
#  dados e configuração
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.DataFrame({
    "Alternativa": [
        "Alumínio",
        "Ligas de alumínio",
        "Aço carbono",
        "Ligas de titânio",
        "Ligas de níquel",
        "Ligas de zinco",
        "Ligas de cobre"
    ],
    "Densidade":          [2700, 2700, 7900, 4500, 8900, 6000, 8930],
    "Custo":              [2.32, 2.30, 0.85, 13.00, 6.00, 2.20, 2.30],
    "Rigidez específica": [0.03, 0.03, 0.03,  0.02, 0.02, 0.02, 0.02],
    "Corrosão":           [4, 4, 3, 5, 5, 3, 5],
    "Desgaste":           [3, 3, 5, 3, 4, 1, 5]
})

criterios = ["Densidade", "Custo", "Rigidez específica", "Corrosão", "Desgaste"]

tipos = {
    "Densidade":          "custo",
    "Custo":              "custo",
    "Rigidez específica": "beneficio",
    "Corrosão":           "beneficio",
    "Desgaste":           "beneficio"
}

dados

,Alternativa,Densidade,Custo,Rigidez específica,Corrosão,Desgaste
0,Alumínio,2700,2.32,0.03,4,3
1,Ligas de alumínio,2700,2.30,0.03,4,3
2,Aço carbono,7900,0.85,0.03,3,5
3,Ligas de titânio,4500,13.00,0.02,5,3
4,Ligas de níquel,8900,6.00,0.02,5,4
5,Ligas de zinco,6000,2.20,0.02,3,1
6,Ligas de cobre,8930,2.30,0.02,5,5


In [27]:
#  matriz AHP
matriz_ahp = np.array([
    [1,   2,   5,   3,   2],
    [1/2, 1,   4,   3,   3],
    [1/5, 1/4, 1,   1/2, 1/2],
    [1/3, 1/3, 2,   1,   1],
    [1/2, 1/3, 2,   1,   1]
], dtype=float)

matriz_ahp_df = pd.DataFrame(matriz_ahp, index=criterios, columns=criterios)
matriz_ahp_df

,Densidade,Custo,Rigidez específica,Corrosão,Desgaste
Densidade,1.000000,2.000000,5.0,3.0,2.0
Custo,0.500000,1.000000,4.0,3.0,3.0
Rigidez específica,0.200000,0.250000,1.0,0.5,0.5
Corrosão,0.333333,0.333333,2.0,1.0,1.0
Desgaste,0.500000,0.333333,2.0,1.0,1.0


In [28]:
# pesos AHP
autovalores, autovetores = np.linalg.eig(matriz_ahp)

print("Autovalores:")
print(autovalores)

print("\nAutovetores:")
print(autovetores)

indice_maior_autovalor = np.argmax(autovalores.real)

lambda_max          = autovalores.real[indice_maior_autovalor]
autovetor_principal = autovetores[:, indice_maior_autovalor].real

pesos_ahp = autovetor_principal / autovetor_principal.sum()

pesos_ahp = pd.Series(
    pesos_ahp,
    index=criterios,
    name="Peso AHP"
)

print("Maior autovalor:")
print(lambda_max)

print("\nPesos AHP:")
print(pesos_ahp)

Autovalores:
[ 5.10136198e+00+0.j         -5.56343299e-02+0.71018374j
 -5.56343299e-02-0.71018374j  4.95334029e-03+0.10346878j
  4.95334029e-03-0.10346878j]

Autovetores:
[[ 0.73208974+0.j          0.77971825+0.j          0.77971825-0.j
   0.57867371+0.j          0.57867371-0.j        ]
 [ 0.57270767+0.j         -0.18163399+0.5464833j  -0.18163399-0.5464833j
   0.05255907-0.37261514j  0.05255907+0.37261514j]
 [ 0.13004696+0.j          0.01294001-0.02515471j  0.01294001+0.02515471j
  -0.24365813+0.01596168j -0.24365813-0.01596168j]
 [ 0.23181573+0.j         -0.09920699-0.00994237j -0.09920699+0.00994237j
   0.08966451+0.51705591j  0.08966451-0.51705591j]
 [ 0.25573385+0.j         -0.11345423-0.19181137j -0.11345423+0.19181137j
   0.13418581-0.41293559j  0.13418581+0.41293559j]]
Maior autovalor:
5.101361979313954

Pesos AHP:
Densidade             0.380822
Custo                 0.297914
Rigidez específica    0.067648
Corrosão              0.120587
Desgaste              0.133029
Name: Peso

In [25]:
# função COPRAS
def copras(dados, criterios, tipos, pesos):

    # --- normalização proporcional ---
    matriz_norm = dados[criterios] / dados[criterios].sum(axis=0)

    # --- matriz ponderada ---
    matriz_pond = matriz_norm * np.array(pesos)

    # --- somas por tipo de critério ---
    criterios_beneficio = [c for c in criterios if tipos[c] == "beneficio"]
    criterios_custo     = [c for c in criterios if tipos[c] == "custo"]

    S_pos = matriz_pond[criterios_beneficio].sum(axis=1)
    S_neg = matriz_pond[criterios_custo].sum(axis=1)

    # --- significância relativa Q ---
    soma_S_neg     = S_neg.sum()
    soma_inv_S_neg = (1 / S_neg).sum()

    Q = S_pos + soma_S_neg / (S_neg * soma_inv_S_neg)

    # --- resultado ---
    resultado = dados[["Alternativa"]].copy().reset_index(drop=True)
    resultado["S+"]      = S_pos.values
    resultado["S-"]      = S_neg.values
    resultado["Q"]       = Q.values
    resultado["Ranking"] = resultado["Q"].rank(
        ascending=False, method="min"
    ).astype(int)

    return resultado.sort_values("Ranking").reset_index(drop=True)

In [29]:
# COPRAS com pesos AHP
resultado_copras = copras(dados, criterios, tipos, pesos_ahp)
resultado_copras

,Alternativa,S+,S-,Q,Ranking
0,Ligas de alumínio,0.045199,0.048351,0.203420,1
1,Alumínio,0.045199,0.048557,0.202750,2
2,Aço carbono,0.052127,0.081008,0.146563,3
3,Ligas de cobre,0.056464,0.105342,0.129086,4
4,Ligas de zinco,0.025976,0.077510,0.124674,5
5,Ligas de níquel,0.050921,0.143116,0.104375,6
6,Ligas de titânio,0.045378,0.174851,0.089131,7


In [30]:
# análise de sensibilidade: Monte Carlo nos pesos
#
# Gera N vetores de pesos aleatórios (distribuição de Dirichlet,
# que garante Σw = 1 e w > 0) e registra o ranking de cada alternativa.

N = 10_000
np.random.seed(42)

amostras_pesos = np.random.dirichlet(np.ones(len(criterios)), size=N)

rankings = np.zeros((N, len(dados)), dtype=int)

for i, pesos_mc in enumerate(amostras_pesos):
    res = copras(dados, criterios, tipos, pesos_mc)
    # reordena pelo índice original para alinhar as alternativas
    res_ordenado = res.set_index("Alternativa").loc[dados["Alternativa"]].reset_index()
    rankings[i] = res_ordenado["Ranking"].values

rankings_df = pd.DataFrame(rankings, columns=dados["Alternativa"].values)
rankings_df.head()

,Alumínio,Ligas de alumínio,Aço carbono,Ligas de titânio,Ligas de níquel,Ligas de zinco,Ligas de cobre
0,3,2,1,7,6,5,4
1,3,2,1,6,5,7,4
2,3,2,1,7,6,5,4
3,2,1,3,6,5,7,4
4,2,1,3,5,6,7,4


In [31]:
# frequência de cada posição por alternativa
freq_ranking = pd.DataFrame(
    index=dados["Alternativa"],
    columns=range(1, len(dados) + 1),
    data=0.0                          # float desde a criação
)

for alt in dados["Alternativa"]:
    contagens = rankings_df[alt].value_counts()
    for pos, cnt in contagens.items():
        freq_ranking.loc[alt, pos] = cnt / N * 100

freq_ranking.columns = [f"#{p}" for p in freq_ranking.columns]
freq_ranking

,#1,#2,#3,#4,#5,#6,#7
Alternativa,,,,,,,
Alumínio,0.00,56.30,23.05,15.55,4.64,0.46,0.00
Ligas de alumínio,56.66,22.95,15.32,4.61,0.46,0.00,0.00
Aço carbono,35.29,3.87,37.96,10.58,4.05,7.26,0.99
Ligas de titânio,0.05,0.05,7.33,4.95,7.42,37.28,42.92
Ligas de níquel,0.00,2.10,3.73,4.09,45.89,38.14,6.05
Ligas de zinco,0.00,0.00,1.29,10.86,23.79,14.03,50.03
Ligas de cobre,8.00,14.73,11.32,49.36,13.75,2.83,0.01
